# Notebook 05: Learning to Rank

## Why This Matters

Retrieval gives you 100–1000 candidates. Ranking decides the order users actually see. The difference between rank 1 and rank 10 in a recommendation carousel is massive — studies consistently show that position 1 gets 5–10x more clicks than position 5. Netflix estimates that a user will abandon browsing if they can't find something interesting in the first 90 seconds — your ranker is responsible for those first few slots. LambdaMART, the dominant algorithm in this space, powered the winning solutions in numerous web search competitions and is used at virtually every major tech company. This notebook builds a full learning-to-rank pipeline.

## Background

### The Three Paradigms of Ranking

**Pointwise**: Treat each item independently. Predict an absolute relevance score per (query, item) pair. Train with regression (predict rating) or classification (predict click/no-click). Simple, but ignores inter-item relationships — items in the same list compete with each other.

**Pairwise**: For each pair of items shown to the same user, predict which is more relevant. The canonical algorithm is **RankNet** (Burges et al., Microsoft 2005), which uses a sigmoid loss on the score difference:

$$P(i \succ j) = \sigma(s_i - s_j), \quad \mathcal{L} = -\bar{P}_{ij} \log P_{ij} - (1-\bar{P}_{ij}) \log (1-P_{ij})$$

**LambdaRank** extends RankNet by weighting each gradient by the change in NDCG if items $i$ and $j$ are swapped. This directly optimizes the ranking metric even though NDCG is not differentiable:

$$\lambda_{ij} = \frac{\partial \mathcal{L}}{\partial s_i} \cdot |\Delta \text{NDCG}_{ij}|$$

**Listwise**: Optimize the full ranked list directly. **LambdaMART** (Wu et al., 2008) combines LambdaRank gradients with Gradient Boosted Trees — it's fast, interpretable, and consistently wins LTR benchmarks. LightGBM and XGBoost both support LambdaMART natively via `objective=lambdarank`.

**ListNet** (Cao et al., 2007) defines a probability distribution over all possible permutations and minimizes KL divergence from the ideal distribution. Elegant but expensive.

### Why LambdaMART Dominates in Practice

1. **Gradient boosted trees** handle heterogeneous features (dense + sparse, different scales) naturally without normalization
2. **Feature importance** is interpretable — you can debug why an item ranks high
3. **Fast training and inference** — trees are decision stumps, inference is $O(\text{depth} \times \text{n\_trees})$
4. **No cold start on features** — works immediately when item features are available

### Feature Engineering for Ranking

The ranker has access to features that the retrieval model doesn't:
- **User-item interaction features**: did the user click/buy this category before? How many times?
- **Item quality features**: CTR on similar users, average rating, recency
- **Position features**: predicted position at retrieval stage (important for position bias correction)
- **Cross features**: user's affinity for item's genre × item's genre quality

In [ ]:
%pip install -q lightgbm numpy pandas scikit-learn matplotlib tqdm

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings, os, requests, zipfile, io, time

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.style.use("seaborn-v0_8-whitegrid")

# ── Load MovieLens 100K ──────────────────────────────────────────────────────
DATA_DIR = "/tmp/ml-100k"
if not os.path.exists(DATA_DIR):
    r = requests.get("https://files.grouplens.org/datasets/movielens/ml-100k.zip", timeout=60)
    zipfile.ZipFile(io.BytesIO(r.content)).extractall("/tmp/")

ratings = pd.read_csv(f"{DATA_DIR}/u.data", sep="\t",
                      names=["user_id", "item_id", "rating", "timestamp"])
movies = pd.read_csv(f"{DATA_DIR}/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1,2] + list(range(5,24)),
                     names=["item_id","title","release_date"]+[f"genre_{i}" for i in range(19)])

# Map genres
genre_names = ["Unknown","Action","Adventure","Animation","Children","Comedy",
               "Crime","Documentary","Drama","Fantasy","Film-Noir","Horror",
               "Musical","Mystery","Romance","Sci-Fi","Thriller","War","Western"]
genre_cols = [f"genre_{i}" for i in range(19)]

print(f"Ratings: {len(ratings):,} | Movies: {len(movies):,}")
print("Ready.")

## 1. Feature Engineering

Good ranking features are the most impactful thing you can do to improve a ranker. The model is table stakes; the features are the leverage.

We'll build three categories of features:
1. **Item features**: global statistics about the item (popularity, mean rating, recency)
2. **User features**: statistics about the user's rating behavior (activity level, mean rating, rating variance)
3. **User-item cross features**: how well this specific user matches this specific item (genre affinity)

In [ ]:
# ── Item-level features ──────────────────────────────────────────────────────
item_stats = ratings.groupby("item_id").agg(
    item_n_ratings = ("rating", "count"),
    item_mean_rating = ("rating", "mean"),
    item_rating_std = ("rating", "std"),
    item_last_rated = ("timestamp", "max"),
    item_first_rated = ("timestamp", "min"),
).reset_index()
item_stats["item_rating_std"] = item_stats["item_rating_std"].fillna(0)

# Popularity rank
item_stats["item_popularity_rank"] = item_stats["item_n_ratings"].rank(pct=True)

# Log-scale popularity (more informative than raw count)
item_stats["item_log_popularity"] = np.log1p(item_stats["item_n_ratings"])

# ── User-level features ───────────────────────────────────────────────────────
user_stats = ratings.groupby("user_id").agg(
    user_n_ratings = ("rating", "count"),
    user_mean_rating = ("rating", "mean"),
    user_rating_std = ("rating", "std"),
    user_rating_min = ("rating", "min"),
    user_rating_max = ("rating", "max"),
).reset_index()
user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0)
user_stats["user_rating_range"] = user_stats["user_rating_max"] - user_stats["user_rating_min"]

# ── Merge all features ────────────────────────────────────────────────────────
df = ratings.merge(item_stats, on="item_id").merge(user_stats, on="user_id")

# Genre features from movie metadata
movies_genres = movies[["item_id"] + genre_cols].copy()
df = df.merge(movies_genres, on="item_id")

# ── User-genre affinity ───────────────────────────────────────────────────────
# For each user, compute mean rating given to each genre
genre_affinity = []
for g_idx, g_col in enumerate(genre_cols):
    genre_items = movies[movies[g_col] == 1]["item_id"].values
    user_genre_ratings = (ratings[ratings.item_id.isin(genre_items)]
                          .groupby("user_id")["rating"]
                          .mean()
                          .rename(f"user_affinity_{g_col}"))
    genre_affinity.append(user_genre_ratings)

user_genre_df = pd.concat(genre_affinity, axis=1).reset_index()
# Fill missing with overall user mean
user_genre_df = user_genre_df.merge(user_stats[["user_id","user_mean_rating"]], on="user_id")
for g_col in genre_cols:
    col = f"user_affinity_{g_col}"
    user_genre_df[col] = user_genre_df[col].fillna(user_genre_df["user_mean_rating"])
user_genre_df = user_genre_df.drop(columns=["user_mean_rating"])
df = df.merge(user_genre_df, on="user_id")

# Item-genre match: how much does user's genre affinity match this item's genres?
affinity_cols = [f"user_affinity_genre_{i}" for i in range(19)]
df["genre_affinity_score"] = sum(
    df[f"user_affinity_genre_{i}"] * df[f"genre_{i}"]
    for i in range(19)
) / (df[genre_cols].sum(axis=1).clip(lower=1))

# Deviation from user mean (normalized rating — captures relative preference)
df["rating_deviation"] = df["rating"] - df["user_mean_rating"]

print(f"Feature matrix shape: {df.shape}")
print(f"Features: {[c for c in df.columns if c not in ['user_id','item_id','rating','timestamp','title','release_date']][:10]}...")

## 2. LTR Dataset Preparation

LambdaMART requires queries and per-query relevance labels. In recommendation, each *user* is a "query" and their candidates are the documents. Relevance = rating (1–5).

LightGBM's `lambdarank` objective expects:
- `group`: array of document counts per query (e.g., [5, 3, 7] means first query has 5 docs)
- `label`: relevance scores (0–4 scale for LightGBM, so we subtract 1)
- Features: numeric feature matrix

In [ ]:
# Feature columns for the ranker
feature_cols = [
    "item_n_ratings", "item_mean_rating", "item_rating_std",
    "item_popularity_rank", "item_log_popularity",
    "user_n_ratings", "user_mean_rating", "user_rating_std",
    "user_rating_range", "genre_affinity_score",
] + genre_cols

# Add affinity features
feature_cols += [f"user_affinity_genre_{i}" for i in range(19)]

X = df[feature_cols].values.astype(np.float32)
y = (df["rating"].values - 1).astype(np.int32)  # 0–4 scale
groups = df["user_id"].values

# Temporal split: train on first 80% of each user's ratings
df = df.sort_values(["user_id", "timestamp"])
train_mask = np.zeros(len(df), dtype=bool)
for uid, group in df.groupby("user_id"):
    idxs = group.index
    cutoff = max(1, int(len(idxs) * 0.8))
    train_mask[idxs[:cutoff]] = True

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[~train_mask], y[~train_mask]
groups_train = groups[train_mask]
groups_test  = groups[~train_mask]

# LightGBM needs group sizes (number of docs per query)
from collections import Counter
def get_group_sizes(grp_arr):
    c = Counter()
    prev = None
    sizes = []
    for g in grp_arr:
        if g != prev:
            if prev is not None:
                sizes.append(c[prev])
            c[g] = 0
            prev = g
        c[g] += 1
    if prev is not None:
        sizes.append(c[prev])
    return sizes

train_group_sizes = get_group_sizes(groups_train)
test_group_sizes  = get_group_sizes(groups_test)

print(f"Train: {X_train.shape[0]:,} examples, {len(train_group_sizes):,} queries")
print(f"Test:  {X_test.shape[0]:,}  examples, {len(test_group_sizes):,}  queries")
print(f"Avg docs per query (train): {np.mean(train_group_sizes):.1f}")

## 3. Training LambdaMART with LightGBM

LambdaMART combines LambdaRank gradients (which directly optimize NDCG) with gradient boosted decision trees (GBDT). LightGBM implements this via `objective=lambdarank`. Key hyperparameters:

- `ndcg_eval_at`: which K values to report NDCG for during training
- `num_leaves`: controls tree complexity — larger = more expressive but slower
- `min_data_in_leaf`: regularization — prevents overfitting on small queries
- `label_gain`: how much each relevance level contributes to NDCG

In [ ]:
# LightGBM LambdaMART
train_data = lgb.Dataset(
    X_train, label=y_train,
    group=train_group_sizes,
    feature_name=feature_cols
)
valid_data = lgb.Dataset(
    X_test, label=y_test,
    group=test_group_sizes,
    feature_name=feature_cols,
    reference=train_data
)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [1, 5, 10],
    "num_leaves": 31,
    "learning_rate": 0.05,
    "n_estimators": 200,
    "min_data_in_leaf": 5,
    "label_gain": [0, 1, 3, 7, 15],  # relevance 0–4 → gain
    "verbose": -1,
    "num_threads": 4,
}

print("Training LambdaMART (LightGBM)...")
t0 = time.time()
evals_result = {}
ranker = lgb.train(
    params,
    train_data,
    num_boost_round=200,
    valid_sets=[valid_data],
    valid_names=["valid"],
    callbacks=[
        lgb.record_evaluation(evals_result),
        lgb.early_stopping(stopping_rounds=20, verbose=False),
        lgb.log_evaluation(period=50),
    ]
)
print(f"Training time: {time.time()-t0:.1f}s | Best iteration: {ranker.best_iteration}")

## 4. NDCG Evaluation

NDCG@K is the gold standard for ranking evaluation. It rewards models that place highly relevant items near the top of the list:

$$\text{NDCG@K} = \frac{1}{|U|} \sum_u \frac{\sum_{k=1}^{K} \frac{2^{\text{rel}_k} - 1}{\log_2(k+1)}}{\text{IDCG@K}}$$

where $\text{rel}_k$ is the relevance of the item at rank $k$ and IDCG is the ideal (perfect) ranking's DCG.

In [ ]:
def ndcg_at_k(y_true, y_score, groups, K):
    """Compute NDCG@K over all queries."""
    ndcgs = []
    start = 0
    for size in groups:
        end = start + size
        if size == 0:
            start = end
            continue
        true_k = y_true[start:end]
        score_k = y_score[start:end]

        # Sort by predicted score
        order = np.argsort(score_k)[::-1][:K]
        rels = true_k[order]

        # DCG
        dcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(rels))

        # IDCG
        ideal = np.sort(true_k)[::-1][:K]
        idcg = sum((2**r - 1) / np.log2(i + 2) for i, r in enumerate(ideal))

        if idcg > 0:
            ndcgs.append(dcg / idcg)
        start = end

    return np.mean(ndcgs)


y_pred_train = ranker.predict(X_train)
y_pred_test  = ranker.predict(X_test)

for K in [1, 5, 10]:
    ndcg_train = ndcg_at_k(y_train, y_pred_train, train_group_sizes, K)
    ndcg_test  = ndcg_at_k(y_test,  y_pred_test,  test_group_sizes,  K)
    print(f"NDCG@{K:2d}  Train: {ndcg_train:.4f}  |  Test: {ndcg_test:.4f}")

# Learning curves
fig, ax = plt.subplots(figsize=(9, 4))
for k_val in [1, 5, 10]:
    key = f"ndcg@{k_val}"
    if key in evals_result.get("valid", {}):
        ax.plot(evals_result["valid"][key], label=f"NDCG@{k_val}")
ax.set_xlabel("Boosting Round")
ax.set_ylabel("NDCG")
ax.set_title("LambdaMART Learning Curves (validation set)")
ax.legend()
plt.tight_layout()
plt.savefig("/tmp/lambdamart_curves.png", dpi=120)
plt.show()

## 5. Feature Importance

One major advantage of GBDT rankers over neural models is interpretability. Feature importance tells us which signals the model relies on most — crucial for debugging, fairness analysis, and feature engineering decisions.

In [ ]:
# Feature importance
importance = pd.DataFrame({
    "feature": feature_cols,
    "gain": ranker.feature_importance(importance_type="gain"),
    "split": ranker.feature_importance(importance_type="split"),
}).sort_values("gain", ascending=False)

print("Top 15 features by gain:")
print(importance.head(15)[["feature", "gain", "split"]].to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gain importance
top_gain = importance.head(15)
axes[0].barh(top_gain.feature[::-1], top_gain.gain[::-1], color="steelblue")
axes[0].set_xlabel("Feature Importance (Gain)")
axes[0].set_title("Top 15 Features by Gain\n(total gain from splits using this feature)")

# Split importance
top_split = importance.sort_values("split", ascending=False).head(15)
axes[1].barh(top_split.feature[::-1], top_split.split[::-1], color="coral")
axes[1].set_xlabel("Feature Importance (Split Count)")
axes[1].set_title("Top 15 Features by Split Count\n(how often feature is used in splits)")

plt.tight_layout()
plt.savefig("/tmp/feature_importance.png", dpi=120)
plt.show()

## 6. Pointwise vs Pairwise vs Listwise Comparison

Let's compare all three ranking paradigms on the same dataset.

In [ ]:
# Pointwise: regression on rating
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

gb_point = lgb.train(
    {"objective": "regression", "num_leaves": 31, "learning_rate": 0.05,
     "n_estimators": 100, "verbose": -1},
    lgb.Dataset(X_train, label=y_train.astype(float)),
    num_boost_round=100,
    callbacks=[lgb.log_evaluation(period=9999)]
)

# Pairwise: lambdarank with fewer trees
ranker_pair = lgb.train(
    {**params, "n_estimators": 100},
    train_data,
    num_boost_round=100,
    callbacks=[lgb.log_evaluation(period=9999)]
)

# Compute NDCG@10 for each
methods = {
    "Pointwise (Regression)": gb_point.predict(X_test),
    "Pairwise/Listwise (LambdaMART)": ranker.predict(X_test),
    "Pairwise/Listwise (LambdaMART, 100 trees)": ranker_pair.predict(X_test),
}

print("\nComparison: Pointwise vs Listwise (LambdaMART)\n")
print(f"{'Method':<42} {'NDCG@1':>8} {'NDCG@5':>8} {'NDCG@10':>8}")
print("-" * 70)
for name, preds in methods.items():
    row = [name]
    for k in [1, 5, 10]:
        row.append(f"{ndcg_at_k(y_test, preds, test_group_sizes, k):.4f}")
    print(f"{row[0]:<42} {row[1]:>8} {row[2]:>8} {row[3]:>8}")

print("\nLambdaMART wins because it directly optimizes NDCG, not a surrogate loss.")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Pointwise | Predict absolute score — simple, ignores inter-item relationships |
| RankNet | Pairwise sigmoid loss on score differences — Microsoft (2005) |
| LambdaRank | Weight RankNet gradients by $\Delta$NDCG — directly optimizes ranking metric |
| LambdaMART | LambdaRank + GBDT — fast, interpretable, dominant in production |
| NDCG@K | Position-weighted precision — standard LTR evaluation metric |
| Feature importance | GBDT provides free interpretability — use for debugging and fairness |
| Group parameter | Each user = one query; group sizes tell LightGBM query boundaries |
| Label gain | Maps ordinal relevance (0–4) to exponential NDCG gains [0,1,3,7,15] |

### Common Pitfalls
- Using regression loss for ranking — predicts rating, not optimal ordering
- Not including position bias features — users click top results more regardless of quality
- Training with too few data points per query — `min_data_in_leaf` prevents overfitting
- Evaluating NDCG on all users equally — users with 1 test item have trivial NDCG@10=1
- Ignoring business constraints — diversity, freshness, legal rules matter beyond pure NDCG